# Few-shot NIDS — Colab driver

This notebook is a thin driver. All the code lives in the `nids_maml` package on
GitHub, so the notebook never needs editing: change a config, not a cell.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Order of operations:
1. Check the GPU and clone the repo
2. Point at the CIC-IDS2017 CSV
3. Smoke test (~1 min) — confirms the pipeline before spending GPU hours
4. Time a single real run, then launch the matrix
5. Analyse and download results

## 1. Environment and code

In [ ]:
import subprocess, torch, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n*** No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun. ***')

In [ ]:
# This repository. It is public, so no token is needed.
REPO_URL = 'https://github.com/ayushexploring/nids-maml.git'
BRANCH = 'main'
REPO_DIR = '/content/nids-maml'

import os, shutil, subprocess

# Step out of the repository before touching it. Deleting or replacing the
# directory the process is standing in leaves it with no working directory,
# and git then fails with exit 128 and no usable message.
os.chdir('/content')

def sh(*args):
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(' '.join(args) + ' failed: ' + r.stdout + r.stderr)
    return r.stdout.strip()

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    sh('git', '-C', REPO_DIR, 'fetch', '--all', '--quiet')
    # reset rather than pull: discards local drift and cannot leave the
    # tree on old code because of a merge conflict.
    sh('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/' + BRANCH)
    sh('git', '-C', REPO_DIR, 'clean', '-fd')
    print('updated existing clone')
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    sh('git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR)
    print('cloned fresh')

os.chdir(REPO_DIR)

# Drop any copy of the package imported before this update, so the label
# inventory cell below does not read a stale LABEL_MAP.
import sys, importlib
for name in [m for m in list(sys.modules) if m.startswith('nids_maml')]:
    del sys.modules[name]
importlib.invalidate_caches()

print('cwd     :', os.getcwd())
print('commit  :', sh('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'))

In [ ]:
!pip install -q pyyaml
!python -m tests.test_correctness
print()
print('Expect: 20 passed, 0 failed')
print('A lower count means the clone is stale -- re-run the cell above')
print('and check that the commit hash matches what you were told.')

The test suite above is the gate. It checks the specific defects that
invalidated the earlier results — leakage between meta-splits, inner steps
collapsing to one, dropout left on at evaluation, attention over a single
token, binary tasks scored with a 5-way head. **If anything fails, stop here.**

## 2. Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Either a single consolidated CSV or a directory of the daily captures.
DATA_PATH = '/content/drive/MyDrive/PhD/cicids2017_cleaned.csv'
RESULTS_DIR = '/content/drive/MyDrive/PhD/nids_results'   # survives disconnects

import os
assert os.path.exists(DATA_PATH), f'not found: {DATA_PATH}'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('data :', DATA_PATH, f'({os.path.getsize(DATA_PATH) / 1e6:.0f} MB)'
      if os.path.isfile(DATA_PATH) else '(directory)')
print('out  :', RESULTS_DIR)

In [ ]:
# Inspect the label inventory before training. Any label printed as unmapped
# needs adding to LABEL_MAP in nids_maml/data.py, or its flows are discarded.
import pandas as pd, os
from nids_maml.data import LABEL_MAP, normalise_label

if os.path.isfile(DATA_PATH):
    labels = pd.read_csv(DATA_PATH, usecols=lambda c: c.strip() in ('Label','label'))
    counts = labels.iloc[:, 0].map(normalise_label).value_counts()
    print(counts.to_string())
    unmapped = [l for l in counts.index if l not in LABEL_MAP]
    print('\nUNMAPPED (these rows would be dropped):', unmapped or 'none')

## 3. Smoke test — synthetic data, about a minute

In [ ]:
!python -m nids_maml.run --config configs/smoke.yaml --output-dir /content/smoke

## 4. One real run, timed

In [ ]:
import time
start = time.time()
!python -m nids_maml.run --config configs/primary.yaml \
    --data-path "$DATA_PATH" --output-dir "$RESULTS_DIR" --seed 0
minutes = (time.time() - start) / 60
print(f'\nsingle run: {minutes:.1f} min  ->  141-run matrix ~{minutes * 141 / 60:.1f} h')

Read the log before continuing:

- **`degenerate class pool` warning** — expected under 5-way over 5 classes.
  It means no class is novel at meta-test time, which bounds what the
  "adapts to unseen attacks" claim can say.
- **Meta-validation accuracy flat at 1/n_way** — the inner loop is not
  bootstrapping. Raise `algorithm.inner_lr`; at 0.01 a Transformer base
  learner stays at chance indefinitely.
- **Meta-train loss falling while meta-validation loss rises** — meta-overfitting.
  Early stopping handles it; the gap itself is a reportable result.

## 5. Tuning sweep — choose the operating point on this data

Eight short runs over the inner and outer learning rates. The inner learning
rate is decisive: below a dataset-dependent threshold the Transformer base
learner does not meta-learn at all and validation accuracy stays at chance.
Picking it here, on the real data, avoids discovering it 141 runs later.


In [ ]:
!python scripts/experiments.py --run --only tuning     --data-path "$DATA_PATH" --output-dir "$RESULTS_DIR"

In [ ]:
!python scripts/pick_config.py --results "$RESULTS_DIR"

If the winning configuration differs from `configs/primary.yaml`, edit
`algorithm.inner_lr` / `algorithm.meta_lr` there and commit, or override per
run. **Send me this table before starting the matrix.**


## 6. The matrix

Writes into Drive and skips runs whose result file already exists, so if the
session drops, reconnect and re-run this cell to resume. Run `baselines`
first — that is the table the reviewers asked for.

In [ ]:
!python scripts/experiments.py --run --only baselines \
    --data-path "$DATA_PATH" --output-dir "$RESULTS_DIR"

In [ ]:
!python scripts/experiments.py --run --only ablations \
    --data-path "$DATA_PATH" --output-dir "$RESULTS_DIR"

## 7. Analysis, tables and figures

In [ ]:
!python -m nids_maml.analyse --results "$RESULTS_DIR" \
    --out "$RESULTS_DIR/paper_assets" --figures

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob(f'{RESULTS_DIR}/paper_assets/*.png')):
    print(path.split('/')[-1])
    display(Image(path))

In [ ]:
# Bundle everything for download, then commit it beside the manuscript.
import shutil
archive = shutil.make_archive('/content/nids_results', 'zip', RESULTS_DIR)
print(archive, f'({os.path.getsize(archive) / 1e6:.1f} MB)')
from google.colab import files
files.download(archive)